# Lesson 8a: Policy Gradient Methods - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Policy-Based Methods** - Direct policy optimization
2. **Master the Policy Gradient Theorem** - Foundation of modern RL
3. **Learn REINFORCE Algorithm** - Monte Carlo policy gradient
4. **Understand Baselines** - Variance reduction techniques
5. **Learn Actor-Critic Methods** - Combining policy gradients with value functions
6. **Master A2C and A3C** - Synchronous and asynchronous actor-critic
7. **Understand GAE** - Generalized Advantage Estimation

Policy gradient methods are the foundation of modern deep RL, leading to PPO, SAC, and more.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. From Value-Based to Policy-Based Methods

### Value-Based Methods (Lessons 1-7)

**Approach**: Learn value function, derive policy

**Q-Learning/DQN**:
1. Learn $Q(s,a)$
2. Policy: $\pi(s) = \arg\max_a Q(s,a)$

**Advantages**:
- ✅ Sample efficient
- ✅ Off-policy (use old data)
- ✅ Well-understood theory

**Limitations**:
- ❌ Discrete actions only (argmax)
- ❌ Policy is deterministic
- ❌ Small changes in Q → large changes in policy (instability)
- ❌ Cannot represent stochastic policies

### Policy-Based Methods

**Approach**: Learn policy directly!

**Parameterized policy**:
$$\pi(a|s; \theta)$$

where $\theta$ are policy parameters (neural network weights)

**Objective**: Maximize expected return
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau)]$$

where $\tau = (s_0, a_0, r_1, s_1, a_1, ...)$ is trajectory

**Optimization**: Gradient ascent
$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

### Advantages of Policy Gradients

**1. Continuous actions**:
- Can output any action in continuous space
- e.g., $\pi(a|s) = \mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$
- Critical for robotics!

**2. Stochastic policies**:
- Naturally represent randomness
- Important for:
  - Rock-paper-scissors (no deterministic optimum)
  - Aliased states (partially observable)
  - Exploration

**3. Better convergence**:
- Small change in $\theta$ → small change in $\pi$
- Smoother optimization landscape
- Often more stable than DQN

**4. Effective in high dimensions**:
- Can learn good policy even when value function complex
- Policy may be simpler than value function

### Disadvantages

**1. Sample inefficiency**:
- Typically on-policy (cannot reuse old data)
- Need many environment interactions
- Slower than DQN in wall-clock time

**2. High variance**:
- Monte Carlo estimates noisy
- Requires variance reduction (baselines, critics)

**3. Local optima**:
- Gradient ascent finds local max
- Not guaranteed global optimum

### When to Use Which?

**Use Value-Based (DQN)**:
- Discrete action space
- Sample efficiency critical
- Deterministic policy okay
- e.g., Atari games

**Use Policy-Based (PG)**:
- Continuous action space
- Need stochastic policy
- High-dimensional actions
- e.g., Robotics, continuous control

**Best of both: Actor-Critic**
- Combines policy gradient with value function
- Reduces variance
- Modern algorithms: PPO, SAC, TD3
- Covered in this lesson!

## 2. The Policy Gradient Theorem

### Objective Function

**Goal**: Maximize expected return under policy $\pi_\theta$

**For episodic tasks**:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T-1} \gamma^t r_t \right] = \mathbb{E}_{\tau \sim \pi_\theta} [G_0]$$

**For continuing tasks**:
$$J(\theta) = \mathbb{E}_{s \sim d^\pi} [V^\pi(s)]$$

where $d^\pi(s)$ is stationary distribution under $\pi$

**Question**: How to compute $\nabla_\theta J(\theta)$?

**Challenge**: Gradient depends on:
- Policy parameters $\theta$ (direct)
- State distribution (indirect, through policy!)

### Policy Gradient Theorem

**Theorem** (Sutton et al., 2000):

$$\nabla_\theta J(\theta) \propto \mathbb{E}_{\pi_\theta} \left[ \sum_{t=0}^{\infty} \nabla_\theta \log \pi_\theta(a_t|s_t) Q^{\pi_\theta}(s_t, a_t) \right]$$

**Simplified** (episodic, start state distribution):
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) G_t \right]$$

**Intuition**:
- $\nabla_\theta \log \pi_\theta(a|s)$: Direction to increase prob of action $a$
- $G_t$ or $Q(s,a)$: How good was action $a$?
- Update: Increase probability of good actions, decrease bad actions

### Log-Derivative Trick

**Key identity**:
$$\nabla_\theta \pi_\theta(a|s) = \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s)$$

**Proof**:
$$\nabla_\theta \log \pi_\theta(a|s) = \frac{\nabla_\theta \pi_\theta(a|s)}{\pi_\theta(a|s)}$$

Therefore:
$$\nabla_\theta \pi_\theta(a|s) = \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s)$$

**Why useful?**
- Converts gradient of probability to gradient of log-probability
- Allows expectation over $\pi_\theta$
- Enables Monte Carlo estimation

### Score Function

$$\nabla_\theta \log \pi_\theta(a|s)$$

is called **score function** or **eligibility vector**

**For Gaussian policy** $\pi(a|s) = \mathcal{N}(\mu_\theta(s), \sigma^2)$:

$$\log \pi_\theta(a|s) = -\frac{(a - \mu_\theta(s))^2}{2\sigma^2} + \text{const}$$

$$\nabla_\theta \log \pi_\theta(a|s) = \frac{(a - \mu_\theta(s))}{\sigma^2} \nabla_\theta \mu_\theta(s)$$

**For Softmax policy** (discrete actions):

$$\pi_\theta(a|s) = \frac{\exp(f_\theta(s,a))}{\sum_{a'} \exp(f_\theta(s,a'))}$$

$$\nabla_\theta \log \pi_\theta(a|s) = \nabla_\theta f_\theta(s,a) - \mathbb{E}_{a' \sim \pi} [\nabla_\theta f_\theta(s,a')]$$

### Monte Carlo Estimation

**Policy gradient**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) G_t \right]$$

**Monte Carlo estimate** (sample N trajectories):
$$\nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) G_t^{(i)}$$

**This is REINFORCE!**

## 3. REINFORCE Algorithm

### Algorithm

**REINFORCE** (Williams, 1992) - Monte Carlo policy gradient

```
Initialize policy parameters θ

For each episode:
  Generate trajectory τ = (s₀, a₀, r₁, s₁, ..., s_T) using πθ
  
  For t = 0 to T-1:
    Compute return: G_t = Σ_{k=t}^{T-1} γ^{k-t} r_{k+1}
    
    Compute gradient: ∇_θ log π_θ(a_t|s_t)
    
    Update: θ ← θ + α G_t ∇_θ log π_θ(a_t|s_t)
```

### Key Properties

**1. Unbiased**:
- Uses true returns $G_t$
- No bootstrapping
- Gradient estimate is unbiased

**2. High variance**:
- Returns vary significantly
- Different trajectories → different gradients
- Slow convergence

**3. On-policy**:
- Must use trajectories from current policy
- Cannot reuse old data
- Sample inefficient

**4. Works for both**:
- Discrete actions (softmax policy)
- Continuous actions (Gaussian policy)

### Example: Softmax Policy

**Network**: $f_\theta(s) \to [\text{logits for each action}]$

**Policy**:
$$\pi_\theta(a|s) = \frac{\exp(f_\theta(s)[a])}{\sum_{a'} \exp(f_\theta(s)[a'])}$$

**Log-probability**:
$$\log \pi_\theta(a|s) = f_\theta(s)[a] - \log \sum_{a'} \exp(f_\theta(s)[a'])$$

**Gradient**: Backprop through neural network $f_\theta$

### Example: Gaussian Policy

**Network**: $f_\theta(s) \to \mu$ (mean action)

**Policy**: $\pi_\theta(a|s) = \mathcal{N}(a; \mu_\theta(s), \sigma^2)$

**Sample action**: $a \sim \mathcal{N}(\mu_\theta(s), \sigma^2)$

**Gradient**:
$$\nabla_\theta \log \pi_\theta(a|s) = \frac{a - \mu_\theta(s)}{\sigma^2} \nabla_\theta \mu_\theta(s)$$

### Variance Problem

**High variance** in gradient estimates:

**Example**:
- Good trajectory: $G = +100$
- Bad trajectory: $G = -100$
- Average: $\bar{G} = 0$
- But variance is huge!

**Consequence**:
- Noisy gradients
- Unstable learning
- Slow convergence
- Requires many samples

**Solution**: Baselines!

## 4. Baselines and Variance Reduction

### The Baseline Trick

**Key insight**: Subtract baseline without changing expected gradient!

**Theorem**:
$$\mathbb{E}_{a \sim \pi_\theta} [\nabla_\theta \log \pi_\theta(a|s) b(s)] = 0$$

for any baseline $b(s)$ that doesn't depend on $a$

**Proof**:
$$\mathbb{E}_{a \sim \pi_\theta} [\nabla_\theta \log \pi_\theta(a|s) b(s)]$$
$$= \sum_a \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s) b(s)$$
$$= \sum_a \nabla_\theta \pi_\theta(a|s) b(s)$$
$$= b(s) \nabla_\theta \sum_a \pi_\theta(a|s)$$
$$= b(s) \nabla_\theta 1 = 0$$

**Modified gradient**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau} \left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) (G_t - b(s_t)) \right]$$

**Same expectation, lower variance!**

### Optimal Baseline

**Best baseline** (minimizes variance):
$$b^*(s) = V^\pi(s)$$

**Intuition**:
- $G_t - V(s_t)$ is **advantage**: how much better than average?
- Positive advantage → increase probability
- Negative advantage → decrease probability
- Zero advantage → no update

### REINFORCE with Baseline

```
Initialize policy πθ and value function Vφ

For each episode:
  Generate trajectory τ using πθ
  
  For t = 0 to T-1:
    G_t = Σ_{k=t}^{T-1} γ^{k-t} r_{k+1}
    
    δ = G_t - V_φ(s_t)  # Advantage estimate
    
    θ ← θ + α δ ∇_θ log π_θ(a_t|s_t)  # Policy update
    φ ← φ + α_v (G_t - V_φ(s_t)) ∇_φ V_φ(s_t)  # Value update
```

**Key changes**:
1. Learn value function $V_\phi(s)$
2. Use $(G_t - V(s_t))$ instead of $G_t$
3. Update both policy and value function

### Advantage Function

$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

**Interpretation**:
- How much better is action $a$ than average?
- $A > 0$: Better than average → increase probability
- $A < 0$: Worse than average → decrease probability
- $A = 0$: Average → no change

**Estimating advantage**:
$$\hat{A}_t = G_t - V(s_t)$$

or (with bootstrapping):
$$\hat{A}_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is **Actor-Critic**!

## 5. Actor-Critic Methods

### Architecture

**Two components**:

**Actor**: Policy $\pi_\theta(a|s)$
- Selects actions
- Updated with policy gradient

**Critic**: Value function $V_\phi(s)$ or $Q_\phi(s,a)$
- Estimates value
- Provides baseline/advantage
- Updated with TD learning

### Algorithm: One-Step Actor-Critic

```
Initialize actor πθ and critic Vφ

For each episode:
  Initialize s
  
  For each step:
    a ~ π_θ(·|s)
    Take a, observe r, s'
    
    # TD error (advantage estimate)
    δ = r + γ V_φ(s') - V_φ(s)
    
    # Critic update (TD learning)
    φ ← φ + α_v δ ∇_φ V_φ(s)
    
    # Actor update (policy gradient)
    θ ← θ + α_π δ ∇_θ log π_θ(a|s)
    
    s ← s'
```

### Why Actor-Critic?

**Compared to REINFORCE**:
- ✅ Lower variance (use critic as baseline)
- ✅ Online learning (update every step)
- ✅ Faster convergence
- ❌ Biased (TD targets)

**Bias-variance trade-off**:
- REINFORCE: Unbiased, high variance
- Actor-Critic: Biased (TD), low variance
- AC typically wins in practice!

### Shared vs Separate Networks

**Separate networks**:
```
Actor:  s → network → π(a|s)
Critic: s → network → V(s)
```

**Shared network** (more common):
```
Input: s
  ↓
[Shared layers]
  ↓
  ↙   ↘
[π head] [V head]
  ↓       ↓
π(a|s)   V(s)
```

**Benefits of sharing**:
- Fewer parameters
- Shared feature learning
- Often faster training

### n-Step Actor-Critic

**Generalization**: Use n-step returns

**n-step return**:
$$G_t^{(n)} = r_t + \gamma r_{t+1} + ... + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$

**Advantage**:
$$A_t^{(n)} = G_t^{(n)} - V(s_t)$$

**Trade-off**:
- Small n: Low variance, high bias
- Large n: High variance, low bias
- Typical: n = 5-20

### Generalized Advantage Estimation (GAE)

**Problem**: How to choose n?

**Solution**: Exponentially-weighted average of all n-step advantages!

**GAE(λ)**:
$$A_t^{\text{GAE}(\lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is TD error

**Special cases**:
- λ = 0: $A_t = \delta_t$ (one-step)
- λ = 1: $A_t = \sum_l \gamma^l \delta_{t+l} = G_t - V(s_t)$ (Monte Carlo)

**Typical**: λ = 0.95-0.99

**Used in**: PPO, TRPO (Lesson 9)

## 6. A2C and A3C

### A3C: Asynchronous Advantage Actor-Critic

**Key innovation** (Mnih et al., 2016): Parallel training!

**Architecture**:
```
Global Network (shared parameters θ, φ)
      ↓
    Copy
      ↓
[Worker 1] [Worker 2] ... [Worker N]
   ↓           ↓               ↓
  Env 1       Env 2          Env N
   ↓           ↓               ↓
Local        Local           Local
gradients    gradients       gradients
      ↘       ↓             ↙
        Global update
```

**Algorithm**:
```
Global: Initialize θ, φ

For each worker (parallel):
  Copy global parameters: θ_local ← θ, φ_local ← φ
  
  For t_max steps:
    Interact with environment
    Accumulate gradients using local parameters
  
  Compute gradients dθ, dφ
  
  # Async update (with lock)
  θ ← θ + α dθ
  φ ← φ + α_v dφ
```

**Benefits**:
- ✅ Parallelization (use multiple CPUs)
- ✅ Decorrelated updates (different workers, different states)
- ✅ No replay buffer needed
- ✅ On-policy but efficient

**Challenges**:
- Complex implementation
- Lock contention
- Different workers may interfere

### A2C: Synchronous A3C

**Simplification**: Wait for all workers

**Algorithm**:
```
Global: Initialize θ, φ

While not converged:
  # All workers in parallel
  For each worker i = 1 to N:
    Run policy for t_max steps
    Compute local gradient dθ_i, dφ_i
  
  # Synchronous update (wait for all)
  dθ = (1/N) Σ dθ_i
  dφ = (1/N) Σ dφ_i
  
  θ ← θ + α dθ
  φ ← φ + α_v dφ
```

**Advantages over A3C**:
- ✅ Simpler implementation
- ✅ No locks needed
- ✅ Better GPU utilization (batch updates)
- ✅ Often faster wall-clock time

**Modern preference**: A2C > A3C

### Implementation Details

**Update frequency**: t_max = 5-20 steps

**Learning rates**:
- Actor: α = 7e-4
- Critic: α_v = α (or slightly higher)

**Entropy regularization**:
$$L = L_{\text{actor}} + c_v L_{\text{critic}} - c_{\text{ent}} H(\pi)$$

where $H(\pi) = -\sum_a \pi(a|s) \log \pi(a|s)$ encourages exploration

**Gradient clipping**: Clip norm to 0.5-5.0

**Shared network**: Common layers + separate heads

## Summary

### Key Concepts

✅ **Policy Gradient Theorem** - Foundation of modern RL  
✅ **REINFORCE** - Monte Carlo policy gradient  
✅ **Baselines** - Variance reduction without bias  
✅ **Actor-Critic** - Policy gradient + value function  
✅ **A2C/A3C** - Parallel training for efficiency  
✅ **GAE** - Generalized advantage estimation  

### Critical Equations

**Policy gradient**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau} \left[ \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) G_t \right]$$

**With baseline**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau} \left[ \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) (G_t - V(s_t)) \right]$$

**Actor-Critic update**:
$$\theta \leftarrow \theta + \alpha \delta \nabla_\theta \log \pi_\theta(a|s)$$
$$\phi \leftarrow \phi + \alpha_v \delta \nabla_\phi V_\phi(s)$$

where $\delta = r + \gamma V(s') - V(s)$

### Algorithm Hierarchy

```
REINFORCE
   ↓
REINFORCE + Baseline
   ↓
Actor-Critic (online)
   ↓
A2C/A3C (parallel)
   ↓
PPO (Lesson 9) ← Most popular 2025!
```

### Practical Recommendations

**Start with**: A2C
- Good balance of performance and simplicity
- Easier than A3C
- Better than REINFORCE

**For production**: PPO (next lesson)
- More stable than A2C
- Better sample efficiency
- Industry standard 2025

**Key hyperparameters**:
- Learning rate: 3e-4 to 7e-4
- GAE λ: 0.95-0.99
- n-step: 5-20
- Entropy coefficient: 0.01

### What's Next

**Lesson 8b**: Implement REINFORCE, Actor-Critic, A2C
- Working code for all algorithms
- CartPole and LunarLander
- Continuous control

**Lesson 9**: PPO and TRPO
- Trust regions
- Clipped objective
- State-of-the-art performance

**Lesson 10**: Continuous Control (DDPG, TD3, SAC)
- Actor-critic for continuous actions
- Robotics applications

---

**Lesson 8a Complete!** 🎉

You now understand policy gradient methods, the foundation of modern deep RL!